# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [13]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided information, the most common issues with loans appear to be related to:\n\n- Errors and discrepancies in loan balances and account information.\n- Difficulties in applying payments correctly, often resulting in interest accruing or payments being applied in a way that prolongs the debt.\n- Mishandling of loan transfers without proper notification to borrowers.\n- Problems with loan servicing, such as unhelpful or predatory repayment practices.\n- Issues with inaccurate reporting on credit reports.\n- Challenges in obtaining accurate information about loan terms, balances, or forgiveness options.\n- Mismanagement and improper handling of loan data, including privacy violations and unauthorized disclosures.\n\nWhile specific data on the single most common issue isn't explicitly provided, a recurring theme is problematic dealing with loan servicers—especially errors in balances, misapplication of payments, and lack of transparency—suggesting that issues related to l

In [14]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans often due to a combination of factors including:\n\n1. Lack of clear communication and notification from loan servicers about when repayments were to resume or if there were changes to the loan transfer process, leading to unintentional delinquency.\n2. Complexity and lack of transparency regarding interest accumulation, particularly when loans are placed into forbearance or deferment, causing interest to continue accruing and increasing the total owed.\n3. Limited or no options to adjust payment plans based on individual financial circumstances, making it difficult for borrowers to afford payments while covering basic living expenses.\n4. Misinformation or insufficient guidance about repayment obligations, loan forgiveness programs, and interest calculations, leading borrowers to be unaware of the true amount owed or the consequences of missed payments.\n5. Administrative errors or mismanagement by loan servicers, such as incorrect reporting of l

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issues with loans involve problems with how lenders or servicers handle the loan, such as disputes over fees, difficulty in applying payments correctly, receiving incorrect or bad information about the loan, and issues with repayment terms or loan validity. \n\nIn particular, issues related to dealing with lenders or servicers—such as miscommunication, incorrect application of payments, or lack of transparency—appear repeatedly. Therefore, the most common issue with loans seems to be problems related to **"Dealing with your lender or servicer."**'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, all the complaints listed have a response status marked as "Closed with explanation," and the responses are indicated as "Timely response?": "Yes" for each. This suggests that the complaints were handled in a timely manner. Therefore, there is no indication that any complaints did not get handled in a timely manner.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including difficulties with payment plans, issues with the way their loans were managed or transferred, lack of communication from the loan servicers, and problems caused by incorrect or unreceived information. Some specific issues noted include being steered into the wrong types of forbearances, not receiving timely or clear notices about loan status or changes, and experiencing technical problems like payments being reversed or not processed correctly. Additionally, inadequate communication from loan servicers and failure to respond to requests for deferments or forbearances also contributed to borrowers' inability to repay their loans."

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to involve problems with dealing with lenders or servicers, including errors, miscommunications, and mishandling of information. Many complaints highlight issues such as receiving incorrect or bad information about the loan, errors in loan balances, misapplied payments, wrongful denials of payment plans, and violations of privacy or data mishandling.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, it appears that at least one complaint was not handled in a timely manner. Specifically, the complaint regarding the student loan account review—submitted over a year prior—has been ongoing without resolution for nearly 18 months. The individual states they have not received any response or resolution despite numerous requests. \n\nHowever, for the complaint about the issue with payments not being applied to the account, the response indicates it was "Closed with explanation" and the response was timely, so that issue appears to have been addressed promptly.\n\nTherefore, yes, there was at least one complaint that was not handled in a timely manner.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a combination of factors including a lack of clear information about repayment obligations, the continuous accumulation of interest, and insufficient communication from lenders or servicers. Some borrowers were unaware that they needed to repay their loans, especially when they were not properly informed by financial aid officers. Others faced challenges because their loan balances grew despite making payments, due to interest accruing and being poorly explained or untracked. Additionally, difficulties in managing payment plans, limited options beyond forbearance or deferment, and the complexities of interest accumulation contributed to repayment problems. Overall, inadequate information, communication issues, and the economic difficulties faced by borrowers hindered their ability to successfully pay back their loans.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and context, the most common issues with loans include:\n\n- Trouble with how payments are being handled, including inability to apply payments correctly or pay off loans more quickly.\n- Errors and inaccuracies in loan balances, interest calculations, and reporting on credit reports.\n- Servicing failures such as lack of proper communication, failed investigation into problems, or wrongful credit reporting.\n- Unfair or unexpected increases in interest rates and balances due to mismanagement or systemic issues.\n- Difficulty accessing or understanding loan information, penalties, or repayment options.\n- Issues related to loan transfers, misapplied payments, or wrongful delinquency reports.\n- Coercive practices like forbearance steering, which limit options and lead to increased debt.\n\nOverall, the most common issue appears to be mishandling and mismanagement by loan servicers, leading to inaccurate information, difficulty in repayment, and damage 

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided, several complaints indicate delays and issues in handling requests in a timely manner. Notably:\n\n- Multiple complaints mention that responses or resolutions have taken over a year or nearly two and a half years, with ongoing unresolved issues.\n- Specific complaints state that the company has failed to respond within the expected time frames or has not responded at all, even after follow-ups.\n- For example, Complaint ID 12975634 (Maximus Federal Services) mentions that it has been nearly 18 months with no resolution.\n- Complaint ID 12709087 (MOHELA) indicates that the response was not timely, although they eventually responded.\n- Similarly, complaints against other loan servicers like EdFinancial Services and Nelnet report delays of several weeks to months.\n\nTherefore, yes, several complaints suggest that a number of issues did not get handled in a timely manner, with delays spanning from weeks to over a year, and some issues remain unresolved

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to factors such as the accumulation of interest during forbearance or deferment periods that negated their payments, lack of proper information about repayment options like income-driven repayment plans, and issues with loan servicing practices like mismanagement, miscommunication, or aggressive collection activities. Many borrowers also experienced systemic failures, such as being misled into long-term forbearances, being coerced into consolidation without being informed of alternatives like rehabilitation or income-driven plans, and facing errors in account reporting, which further hindered their ability to repay. Financial hardships, stagnant wages, and economic conditions also contributed to their inability to meet repayment obligations, especially when the available options extended the repayment period and increased overall debt due to interest capitalization.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:

Recall is defined as:

Recall = TP / (TP + FN)
 
where TP (true positives) are the relevant documents that were successfully retrieved, and FN (false negatives) are the relevant documents that were not retrieved.

Recall represents the fraction of all relevant documents that were actually retrieved. It measures how well the system captures the complete set of relevant information.

With this in mind, if we increase the number of retrievals, for example by using query reformulation (such as multi-query retrievers), we introduce semantic variety into the retrieval process. This increases the chance of retrieving additional relevant documents that a single query might miss. As a result, the LLM is more likely to receive a more complete and truly relevant context, which can lead to improved recall and better downstream answer quality.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the context provided, appears to be problems related to federal student loan servicing, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues with the accuracy of credit reporting such as incorrect information on credit reports and unverified debt. Many complaints also involve confusion over interest rates, loan transfers, and improper collection practices.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided, several complaints were not handled in a timely manner. Specifically, the complaints related to student loan servicing by MOHELA (Rows 441 and 84) indicate that the responses were "No" for timely response, and the complainant waited for extended periods without hearing back. Additionally, the complaint regarding Aidvantage (Row 418) was responded to "Yes" for timely response, but the issue remains unresolved, and the complainant reports ongoing problems.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of mismanagement, lack of proper information, and unforeseen financial hardships. In some cases, loan servicers resumed payments unexpectedly during periods when borrowers were still in school or during grace periods, without clear explanations, which led to delinquency. Others faced severe financial difficulties after graduation, including unemployment or underemployment, high interest accruing from deferments and forbearance, and a lack of transparency or support from institutions. Additionally, issues such as misrepresentation by educational institutions about the value and stability of their programs, as well as administrative errors like incorrect reporting or failure to notify borrowers about repayment obligations, also contributed to difficulties in repayment.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, a common issue with loans appears to be problems related to "Dealing with your lender or servicer," such as errors in loan balances, misapplied payments, wrongful denials of repayment plans, and poor communication. Many complaints mention incorrect information about loan status, improper handling of repayment options, lack of notification about loan modifications, and mismanagement of loan classifications.\n\nIn particular, the most frequent issues involve:\n- Errors or inaccuracies in loan balances and payment histories.\n- Misreporting loan statuses (e.g., reporting loans as delinquent or in default without proper notice).\n- Lack of clear communication or notification from loan servicers regarding changes, balances, or repayment options.\n- Improper handling or misclassification of loans (such as incorrect loan type or status).\n\nTherefore, the most common issue with loans, as indicated by these complaints, is **problems with loan servicing, 

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints and responses, it appears that many complaints were not handled in a timely manner. Specifically, the following instances indicate delays:\n\n- Complaint ID 12709087 (submitted 03/28/25 to MOHELA): Response was delayed; the response was "Closed with explanation," and it was marked as "Disputed" or unresolved after over a month.\n\n- Complaint ID 12935889 (submitted 04/11/25 to MOHELA): Marked as "Response was late" or "No response," indicating response timing issues.\n\n- Complaint ID 12739706 (submitted 04/01/25 to MOHELA): Marked "No" for timely response.\n\n- Complaint ID 13056764 (submitted 04/18/25 to EdFinancial Services): Response was "Closed with explanation," and marked as "Timely response? Yes", but other complaints show delayed responses.\n\n- Multiple complaints about inaccurate reporting, collection issues, or unresolved cases (e.g., complaint IDs 13091395, 13117223, 13062402, etc.) also suggest that many issues remain unresolved or delaye

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. **Lack of information and communication**: Many borrowers were not adequately informed about their repayment obligations, the resumption of payments, or changes in loan servicers and transfer notices. This led to unintentional delinquencies and damage to credit scores.\n\n2. **Rising interest and compounding penalties**: Interest continued to accrue, especially during deferment or forbearance, causing the total debt to balloon over time. Borrowers often found their balances increasing despite making payments, due to high interest rates and improper management.\n\n3. **Financial hardships and economic conditions**: Many borrowers experienced financial difficulties, low wages, or unemployment, making regular payments unaffordable or impossible without relief options.\n\n4. **Mismanagement and unethical practices by servicers**: Tactics such as steering borrowers into long-term forbearances, failing to offer inco

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data)

Let's create a new vector store.

In [46]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [47]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [48]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [49]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issue with loans, particularly student loans, appears to be related to problems with dealing with lenders or servicers. This includes issues such as errors in loan balances, misapplied payments, wrongful denials of payment plans, disagreements over interest rates and balances, inaccurate or incomplete information, and poor communication or customer service. Many complaints highlight discrepancies in account records, unauthorized transfers, unfair or confusing practices, and systemic breakdowns in loan management and reporting.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided complaints, yes, several complaints indicate that issues were not handled in a timely manner. For example:\n\n- Complaint assigned ID '12832400' mentions that Aidvantage failed to respond despite the CFPB's 15-day response policy, and the issue remains unresolved.\n- Complaint ID '1249ac2564944e61b978d38d0503139f' states the complainant waited over 2-3 weeks with no resolution.\n- Other complaints also mention delays, lack of responses, and unresolved issues despite multiple follow-ups.\n\nTherefore, the data suggests that some complaints did not get handled promptly or within reasonable timeframes."

In [52]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of factors explained in the complaints. These include:\n\n1. **Accumulation of Interest**: Many borrowers were not adequately informed that interest would continue to accrue and compound during forbearance or deferment, which increased their overall debt and made repayment more difficult over time.\n\n2. **Inadequate or Misleading Information**: Borrowers often reported that loan servicers failed to provide clear guidance on repayment options, including income-based repayment plans or the impact of forbearance, leaving them unprepared and in difficult financial situations.\n\n3. **Systemic and Servicer Failures**: Some complaints cited issues such as loans being transferred between multiple servicers without proper notification, incorrect or outdated account information, and misreported payments or delinquencies, which negatively affected credit scores and repayment ability.\n\n4. **Financial Hardships and Economic F

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE